In [1]:
!pip install -q wandb sentence-transformers scikit-learn

import wandb
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from kaggle_secrets import UserSecretsClient

# Load W&B API key from Kaggle Secrets - no manual login needed
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

run = wandb.init(
    entity="23f2005025-dl-genai-project",
    project="dl-genai-project",
    name="day1-tfidf-baseline",
    job_type="baseline"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: meetbatra (meetbatra-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 1fwubbwp
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260725_152533-1fwubbwp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run day1-tfidf-baseline
wandb: ⭐️ View project at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/r

In [2]:
# Load competition data
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train.head())

# Create a local validation split from train (80/20)
# so we can measure mAP@3 ourselves before submitting
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)
print("Train split:", train_split.shape)
print("Val split:", val_split.shape)

Train shape: (2000, 8)
Test shape: (500, 7)
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that 

In [3]:
def apk(actual, predicted, k=3):
    """Average precision at k for a single prediction"""
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
            break  # only one correct answer possible per question
    return score

def mapk(actual, predicted, k=3):
    """Mean average precision at k"""
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])


def tfidf_predict_top3(train_df, val_df, option_cols=['A', 'B', 'C', 'D', 'E']):
    """For each question, rank options by TF-IDF cosine similarity to the prompt"""
    predictions = []

    for idx, row in val_df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        # Fit TF-IDF on prompt + all options for this question
        corpus = [prompt] + options
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(corpus)

        prompt_vec = tfidf_matrix[0:1]
        option_vecs = tfidf_matrix[1:]

        sims = cosine_similarity(prompt_vec, option_vecs)[0]

        # Rank option letters by similarity, descending
        ranked_idx = np.argsort(sims)[::-1]
        ranked_letters = [option_cols[i] for i in ranked_idx]

        predictions.append(ranked_letters[:3])

    return predictions


# Run baseline on validation set
val_predictions = tfidf_predict_top3(train_split, val_split)
val_actual = val_split['answer'].tolist()

score = mapk(val_actual, val_predictions, k=3)
print(f"TF-IDF baseline local mAP@3: {score:.4f}")

wandb.log({"local_map3": score, "approach": "tfidf_baseline"})

TF-IDF baseline local mAP@3: 0.3121


In [4]:
from sentence_transformers import SentenceTransformer

# Load a lightweight but strong sentence embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

def embedding_predict_top3(val_df, model, option_cols=['A', 'B', 'C', 'D', 'E']):
    predictions = []

    for idx, row in val_df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        prompt_emb = model.encode([prompt])
        option_embs = model.encode(options)

        sims = cosine_similarity(prompt_emb, option_embs)[0]

        ranked_idx = np.argsort(sims)[::-1]
        ranked_letters = [option_cols[i] for i in ranked_idx]

        predictions.append(ranked_letters[:3])

    return predictions


val_predictions_emb = embedding_predict_top3(val_split, model)
score_emb = mapk(val_actual, val_predictions_emb, k=3)

print(f"MiniLM embedding local mAP@3: {score_emb:.4f}")

wandb.log({"local_map3": score_emb, "approach": "minilm_embeddings"})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

MiniLM embedding local mAP@3: 0.3996


In [5]:
# Build a legitimate knowledge base: use ALL options (not just correct answers)
# from train, so retrieval has to genuinely discriminate, not cheat via answer key.
# Each doc is tagged with its source row + option letter for traceability.

kb_docs = []
kb_metadata = []  # (row_id, option_letter) for each doc

for idx, row in train.iterrows():
    for opt in ['A', 'B', 'C', 'D', 'E']:
        kb_docs.append(row[opt])
        kb_metadata.append((row['id'], opt))

print(f"Knowledge base size: {len(kb_docs)} documents")
print(f"Example doc: {kb_docs[0]}")

Knowledge base size: 10000 documents
Example doc: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.


In [6]:
!pip install -q faiss-cpu

import faiss

# Encode all KB docs with the same MiniLM model already loaded in Cell 3
print("Encoding knowledge base...")
kb_embeddings = model.encode(kb_docs, show_progress_bar=True, batch_size=64, convert_to_numpy=True)

# Build FAISS index (L2 distance, same as Milestone 3)
dimension = kb_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(kb_embeddings.astype('float32'))

print(f"FAISS index built: {faiss_index.ntotal} vectors, dim={dimension}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.2 MB/s eta 0:00:00
Encoding knowledge base...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

FAISS index built: 10000 vectors, dim=384


In [7]:
from sentence_transformers import CrossEncoder
from transformers import pipeline as hf_pipeline

# Load cross-encoder for reranking (same as Milestone 3)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Load zero-shot classifier (same as Milestone 3)
zero_shot = hf_pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

def rag_predict_top3(df, option_cols=['A', 'B', 'C', 'D', 'E'], retrieve_k=10, rerank_top_n=3, exclude_leakage=False):
    """
    For each question:
    1. Retrieve top-k candidate docs via FAISS bi-encoder
    2. Rerank with cross-encoder, take top rerank_top_n
    3. Concatenate reranked context + prompt as the zero-shot premise
    4. Score each option as a candidate label, rank by confidence
    """
    predictions = []

    for idx, row in df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        # Step 1: bi-encoder retrieval
        query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
        distances, indices = faiss_index.search(query_emb, retrieve_k)

        # Exclude docs sourced from this exact row (avoid leakage when running on train data)
        candidate_idxs = [i for i in indices[0] if kb_metadata[i][0] != row['id']] if exclude_leakage else list(indices[0])
        candidate_docs = [kb_docs[i] for i in candidate_idxs]

        if not candidate_docs:
            candidate_docs = [kb_docs[i] for i in indices[0]]

        # Step 2: cross-encoder rerank
        pairs = [[prompt, doc] for doc in candidate_docs]
        rerank_scores = cross_encoder.predict(pairs)
        top_context_idxs = np.argsort(rerank_scores)[::-1][:rerank_top_n]
        context = " ".join([candidate_docs[i] for i in top_context_idxs])

        # Step 3: augment prompt with retrieved context
        augmented_premise = f"Context: {context} Question: {prompt}"

        # Step 4: zero-shot classify each option as a candidate label
        result = zero_shot(augmented_premise, options, multi_label=True)

        # Rank options by confidence, map back to letters
        label_to_letter = {row[col]: col for col in option_cols}
        ranked = sorted(zip(result['labels'], result['scores']), key=lambda x: -x[1])
        top3_letters = [label_to_letter[label] for label, score in ranked[:3]]

        predictions.append(top3_letters)

    return predictions

print("RAG pipeline function defined.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

RAG pipeline function defined.


In [8]:
import time

sample_size = 30
val_sample = val_split.head(sample_size).copy()

print(f"Running RAG pipeline on {sample_size} validation rows...")
start = time.time()

rag_predictions = rag_predict_top3(val_sample, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {sample_size} rows ({elapsed/sample_size:.2f}s/row)")

rag_map3 = mapk(val_sample['answer'].tolist(), rag_predictions)
print(f"RAG pipeline local mAP@3 on sample: {rag_map3:.4f}")
print(f"(MiniLM baseline was 0.3996 on full val set)")

Running RAG pipeline on 30 validation rows...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Took 8.7s for 30 rows (0.29s/row)
RAG pipeline local mAP@3 on sample: 0.4389
(MiniLM baseline was 0.3996 on full val set)


In [9]:
print("Running RAG pipeline on full validation split (400 rows)...")
start = time.time()

rag_predictions_full = rag_predict_top3(val_split, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_full = mapk(val_split['answer'].tolist(), rag_predictions_full)
print(f"RAG pipeline local mAP@3 (full val split): {rag_map3_full:.4f}")

Running RAG pipeline on full validation split (400 rows)...
Took 124.1s for 400 rows (0.31s/row)
RAG pipeline local mAP@3 (full val split): 0.4929


In [10]:
kb_docs_v2 = []
kb_metadata_v2 = []

for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb_docs_v2.append(row[correct_letter])
    kb_metadata_v2.append((row['id'], correct_letter))

print(f"New KB size: {len(kb_docs_v2)} documents (was {len(kb_docs)})")

print("Encoding cleaner knowledge base...")
kb_embeddings_v2 = model.encode(kb_docs_v2, show_progress_bar=True, batch_size=64, convert_to_numpy=True)

faiss_index_v2 = faiss.IndexFlatL2(kb_embeddings_v2.shape[1])
faiss_index_v2.add(kb_embeddings_v2.astype('float32'))

print(f"New FAISS index built: {faiss_index_v2.ntotal} vectors")

New KB size: 2000 documents (was 10000)
Encoding cleaner knowledge base...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

New FAISS index built: 2000 vectors


In [11]:
def rag_predict_top3_v2(df, option_cols=['A', 'B', 'C', 'D', 'E'], retrieve_k=10, rerank_top_n=3, exclude_leakage=False):
    """
    Same pipeline as v1, but retrieves from the clean KB (kb_docs_v2 / faiss_index_v2)
    built from correct answers only, instead of all options.
    """
    predictions = []

    for idx, row in df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        # Step 1: bi-encoder retrieval from clean KB
        query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
        distances, indices = faiss_index_v2.search(query_emb, retrieve_k)

        candidate_idxs = [i for i in indices[0] if kb_metadata_v2[i][0] != row['id']] if exclude_leakage else list(indices[0])
        candidate_docs = [kb_docs_v2[i] for i in candidate_idxs]

        if not candidate_docs:
            candidate_docs = [kb_docs_v2[i] for i in indices[0]]

        # Step 2: cross-encoder rerank
        pairs = [[prompt, doc] for doc in candidate_docs]
        rerank_scores = cross_encoder.predict(pairs)
        top_context_idxs = np.argsort(rerank_scores)[::-1][:rerank_top_n]
        context = " ".join([candidate_docs[i] for i in top_context_idxs])

        # Step 3: augment prompt with retrieved context
        augmented_premise = f"Context: {context} Question: {prompt}"

        # Step 4: zero-shot classify each option
        result = zero_shot(augmented_premise, options, multi_label=True)

        label_to_letter = {row[col]: col for col in option_cols}
        ranked = sorted(zip(result['labels'], result['scores']), key=lambda x: -x[1])
        top3_letters = [label_to_letter[label] for label, score in ranked[:3]]

        predictions.append(top3_letters)

    return predictions

print("Running RAG v2 (clean KB) on full validation split (400 rows)...")
start = time.time()

rag_predictions_v2 = rag_predict_top3_v2(val_split, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_v2 = mapk(val_split['answer'].tolist(), rag_predictions_v2)
print(f"RAG v2 (clean KB) local mAP@3: {rag_map3_v2:.4f}")
print(f"(RAG v1 noisy KB: 0.4929, MiniLM baseline: 0.3996)")

Running RAG v2 (clean KB) on full validation split (400 rows)...
Took 137.4s for 400 rows (0.34s/row)
RAG v2 (clean KB) local mAP@3: 0.8767
(RAG v1 noisy KB: 0.4929, MiniLM baseline: 0.3996)


In [12]:
# Check how many val questions have a near-identical twin elsewhere in train
# (same or very similar prompt), which could inflate the RAG score artificially

from difflib import SequenceMatcher

def is_near_duplicate(p1, p2, threshold=0.85):
    return SequenceMatcher(None, p1, p2).ratio() > threshold

duplicate_count = 0
sample_check = val_split.head(50)  # check first 50 for speed

for idx, row in sample_check.iterrows():
    prompt = row['prompt']
    for _, other_row in train.iterrows():
        if other_row['id'] != row['id'] and is_near_duplicate(prompt, other_row['prompt']):
            duplicate_count += 1
            break

print(f"Near-duplicate questions found: {duplicate_count} / {len(sample_check)}")

Near-duplicate questions found: 39 / 50


In [13]:
# The real question: do TEST prompts have near-duplicate matches in TRAIN?
# If yes, RAG retrieval genuinely helps on the leaderboard too, not just in-sample.

test_sample = test.head(30)
test_duplicate_count = 0

for idx, row in test_sample.iterrows():
    prompt = row['prompt']
    for _, train_row in train.iterrows():
        if is_near_duplicate(prompt, train_row['prompt']):
            test_duplicate_count += 1
            break

print(f"Test questions with a near-duplicate in train: {test_duplicate_count} / {len(test_sample)}")

Test questions with a near-duplicate in train: 29 / 30


In [14]:
# Day 1 baseline: MiniLM embeddings (kept for experiment record, does NOT write final submission)
test_predictions_day1 = embedding_predict_top3(test, model)
submission_day1_reference = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(pred) for pred in test_predictions_day1]
})
print("Day 1 MiniLM approach (reference only, not final submission):")
print(submission_day1_reference.head(10))

Day 1 MiniLM approach (reference only, not final submission):
   ID Prediction
0   1      B E A
1   2      B D C
2   3      A C D
3   4      A E C
4   5      B C D
5   6      E A D
6   7      E C A
7   8      D A C
8   9      A C E
9  10      C E D


In [15]:
# Diagnose: for each val row, did the clean-KB retrieval actually surface
# the "true duplicate" (a train row with a near-identical prompt) in top-k?
# This tells us if the gap is a RETRIEVAL problem or a CLASSIFICATION problem.

hit_at_k_results = []

for idx, row in val_split.iterrows():
    prompt = row['prompt']
    correct_answer_text = row[row['answer']]

    query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
    distances, indices = faiss_index_v2.search(query_emb, 10)

    retrieved_docs = [kb_docs_v2[i] for i in indices[0]]

    # Was the TRUE answer text (or something matching it) in the top-10 retrieved?
    hit = any(is_near_duplicate(correct_answer_text, doc, threshold=0.85) for doc in retrieved_docs)
    hit_at_k_results.append(hit)

hit_rate = sum(hit_at_k_results) / len(hit_at_k_results)
print(f"Retrieval hit-rate @10 on val_split: {hit_rate:.2%}")
print(f"({sum(hit_at_k_results)} / {len(hit_at_k_results)} rows had the true answer text retrievable)")

Retrieval hit-rate @10 on val_split: 76.50%
(306 / 400 rows had the true answer text retrievable)


In [16]:
# Test hit-rate @ k for a few k values to find the point of diminishing returns
for k in [10, 20, 30, 50]:
    hits = 0
    for idx, row in val_split.iterrows():
        prompt = row['prompt']
        correct_answer_text = row[row['answer']]

        query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
        distances, indices = faiss_index_v2.search(query_emb, k)
        retrieved_docs = [kb_docs_v2[i] for i in indices[0]]

        if any(is_near_duplicate(correct_answer_text, doc, threshold=0.85) for doc in retrieved_docs):
            hits += 1

    print(f"Hit-rate @{k}: {hits/len(val_split):.2%}")

Hit-rate @10: 76.50%
Hit-rate @20: 80.50%
Hit-rate @30: 83.50%
Hit-rate @50: 85.00%


In [17]:
print("Running RAG v2 (tuned: k=30, rerank_top_n=5) on full validation split...")
start = time.time()

rag_predictions_tuned = rag_predict_top3_v2(val_split, retrieve_k=30, rerank_top_n=5, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_tuned = mapk(val_split['answer'].tolist(), rag_predictions_tuned)
print(f"RAG tuned (k=30, rerank_top_n=5) local mAP@3: {rag_map3_tuned:.4f}")
print(f"(Previous: k=10/rerank=3 gave 0.8767 local / 0.6956 leaderboard)")

Running RAG v2 (tuned: k=30, rerank_top_n=5) on full validation split...
Took 196.1s for 400 rows (0.49s/row)
RAG tuned (k=30, rerank_top_n=5) local mAP@3: 0.8646
(Previous: k=10/rerank=3 gave 0.8767 local / 0.6956 leaderboard)


In [18]:
print("Running RAG v2 (k=30, rerank_top_n=1 — strict single best context) on full validation split...")
start = time.time()

rag_predictions_strict = rag_predict_top3_v2(val_split, retrieve_k=30, rerank_top_n=1, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_strict = mapk(val_split['answer'].tolist(), rag_predictions_strict)
print(f"RAG (k=30, rerank_top_n=1) local mAP@3: {rag_map3_strict:.4f}")
print(f"(k=10/rerank=3: 0.8767 local/0.6956 LB | k=30/rerank=5: 0.8646 local)")

Running RAG v2 (k=30, rerank_top_n=1 — strict single best context) on full validation split...
Took 100.1s for 400 rows (0.25s/row)
RAG (k=30, rerank_top_n=1) local mAP@3: 0.8642
(k=10/rerank=3: 0.8767 local/0.6956 LB | k=30/rerank=5: 0.8646 local)


In [19]:
from sklearn.metrics.pairwise import cosine_similarity

def rag_predict_top3_v3(df, option_cols=['A', 'B', 'C', 'D', 'E'], retrieve_k=30, rerank_top_n=1, exclude_leakage=False):
    """
    Same retrieve + rerank as v2, but replaces zero-shot classification with
    direct embedding similarity between each option and the best retrieved context.
    """
    predictions = []

    for idx, row in df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
        distances, indices = faiss_index_v2.search(query_emb, retrieve_k)

        candidate_idxs = [i for i in indices[0] if kb_metadata_v2[i][0] != row['id']] if exclude_leakage else list(indices[0])
        candidate_docs = [kb_docs_v2[i] for i in candidate_idxs]

        if not candidate_docs:
            candidate_docs = [kb_docs_v2[i] for i in indices[0]]

        pairs = [[prompt, doc] for doc in candidate_docs]
        rerank_scores = cross_encoder.predict(pairs)
        top_context_idxs = np.argsort(rerank_scores)[::-1][:rerank_top_n]
        best_context = candidate_docs[top_context_idxs[0]]

        # Direct similarity: embed the context doc and each option, rank by cosine sim
        context_emb = model.encode([best_context], convert_to_numpy=True)
        option_embs = model.encode(options, convert_to_numpy=True)

        sims = cosine_similarity(context_emb, option_embs)[0]
        ranked_idxs = np.argsort(sims)[::-1][:3]
        top3_letters = [option_cols[i] for i in ranked_idxs]

        predictions.append(top3_letters)

    return predictions

print("Running RAG v3 (embedding similarity, no zero-shot) on full validation split...")
start = time.time()

rag_predictions_v3 = rag_predict_top3_v3(val_split, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_v3 = mapk(val_split['answer'].tolist(), rag_predictions_v3)
print(f"RAG v3 (embedding similarity) local mAP@3: {rag_map3_v3:.4f}")
print(f"(v2 zero-shot variants were all ~0.86 local / 0.6956 leaderboard)")

Running RAG v3 (embedding similarity, no zero-shot) on full validation split...
Took 19.3s for 400 rows (0.05s/row)
RAG v3 (embedding similarity) local mAP@3: 0.8371
(v2 zero-shot variants were all ~0.86 local / 0.6956 leaderboard)


In [20]:
def rag_predict_top3_v4(df, option_cols=['A', 'B', 'C', 'D', 'E'], retrieve_k=30, rerank_top_n=1, exclude_leakage=False, zs_weight=0.5):
    """
    Ensemble: combine zero-shot classifier confidence with embedding cosine similarity
    for each option, then rank by weighted average.
    """
    predictions = []

    for idx, row in df.iterrows():
        prompt = row['prompt']
        options = [row[col] for col in option_cols]

        query_emb = model.encode([prompt], convert_to_numpy=True).astype('float32')
        distances, indices = faiss_index_v2.search(query_emb, retrieve_k)

        candidate_idxs = [i for i in indices[0] if kb_metadata_v2[i][0] != row['id']] if exclude_leakage else list(indices[0])
        candidate_docs = [kb_docs_v2[i] for i in candidate_idxs]

        if not candidate_docs:
            candidate_docs = [kb_docs_v2[i] for i in indices[0]]

        pairs = [[prompt, doc] for doc in candidate_docs]
        rerank_scores = cross_encoder.predict(pairs)
        top_context_idxs = np.argsort(rerank_scores)[::-1][:rerank_top_n]
        best_context = candidate_docs[top_context_idxs[0]]

        # Signal 1: zero-shot classification score per option
        augmented_premise = f"Context: {best_context} Question: {prompt}"
        zs_result = zero_shot(augmented_premise, options, multi_label=True)
        zs_scores = {label: score for label, score in zip(zs_result['labels'], zs_result['scores'])}
        zs_scores_ordered = np.array([zs_scores[opt] for opt in options])

        # Signal 2: embedding cosine similarity per option
        context_emb = model.encode([best_context], convert_to_numpy=True)
        option_embs = model.encode(options, convert_to_numpy=True)
        sim_scores = cosine_similarity(context_emb, option_embs)[0]

        # Normalize both to 0-1 range, then combine
        zs_norm = (zs_scores_ordered - zs_scores_ordered.min()) / (zs_scores_ordered.max() - zs_scores_ordered.min() + 1e-8)
        sim_norm = (sim_scores - sim_scores.min()) / (sim_scores.max() - sim_scores.min() + 1e-8)

        combined = zs_weight * zs_norm + (1 - zs_weight) * sim_norm
        ranked_idxs = np.argsort(combined)[::-1][:3]
        top3_letters = [option_cols[i] for i in ranked_idxs]

        predictions.append(top3_letters)

    return predictions

print("Running RAG v4 (ensemble: zero-shot + embedding similarity) on full validation split...")
start = time.time()

rag_predictions_v4 = rag_predict_top3_v4(val_split, exclude_leakage=True)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(val_split)} rows ({elapsed/len(val_split):.2f}s/row)")

rag_map3_v4 = mapk(val_split['answer'].tolist(), rag_predictions_v4)
print(f"RAG v4 (ensemble) local mAP@3: {rag_map3_v4:.4f}")
print(f"(v2 zero-shot: ~0.86 local/0.6956 LB | v3 embedding-only: 0.8371 local)")

Running RAG v4 (ensemble: zero-shot + embedding similarity) on full validation split...
Took 104.4s for 400 rows (0.26s/row)
RAG v4 (ensemble) local mAP@3: 0.8650
(v2 zero-shot: ~0.86 local/0.6956 LB | v3 embedding-only: 0.8371 local)


In [21]:
# BiLSTM prep — Step 1: check sequence length distribution before choosing max_len
# This tells us how long "prompt + option" pairs are, so we don't truncate the
# key differentiating clause between similar options (seen in Heidegger-style questions)

def combine_text(prompt, option):
    return f"{prompt} [SEP] {option}"

all_texts = []
for df in [train, test]:
    for _, row in df.iterrows():
        for col in ['A', 'B', 'C', 'D', 'E']:
            all_texts.append(combine_text(row['prompt'], row[col]))

lengths = [len(t.split()) for t in all_texts]

import numpy as np
print(f"Total sequences: {len(lengths)}")
print(f"Mean length: {np.mean(lengths):.1f} words")
print(f"Median length: {np.median(lengths):.1f} words")
print(f"95th percentile: {np.percentile(lengths, 95):.1f} words")
print(f"99th percentile: {np.percentile(lengths, 99):.1f} words")
print(f"Max length: {max(lengths)} words")

Total sequences: 12500
Mean length: 44.9 words
Median length: 41.0 words
95th percentile: 81.0 words
99th percentile: 102.0 words
Max length: 149 words


In [22]:
# BiLSTM prep — Step 2: build vocabulary from train + test text
# From-scratch requirement: no pretrained tokenizer or embeddings, just our own word->index mapping

from collections import Counter
import re

def tokenize(text):
    # simple whitespace + punctuation-aware tokenizer, lowercased
    text = text.lower()
    tokens = re.findall(r"\w+", text)
    return tokens

# Build vocab from ALL prompt+option text (train + test), so no OOV issues at inference
word_counts = Counter()
for df in [train, test]:
    for _, row in df.iterrows():
        for col in ['A', 'B', 'C', 'D', 'E']:
            tokens = tokenize(combine_text(row['prompt'], row[col]))
            word_counts.update(tokens)

print(f"Unique words before filtering: {len(word_counts)}")

# Keep words appearing at least twice — drops rare typos/noise, keeps vocab manageable
MIN_FREQ = 2
vocab_words = [w for w, c in word_counts.items() if c >= MIN_FREQ]
print(f"Vocab size after min_freq={MIN_FREQ}: {len(vocab_words)}")

# Special tokens: 0 = PAD, 1 = UNK
word2idx = {"<PAD>": 0, "<UNK>": 1}
for w in vocab_words:
    word2idx[w] = len(word2idx)

idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(word2idx)
MAX_LEN = 90

print(f"Final vocab size: {VOCAB_SIZE}")

Unique words before filtering: 2981
Vocab size after min_freq=2: 2981
Final vocab size: 2983


In [23]:
# BiLSTM prep — Step 3: encode sequences to padded index tensors, and build the Dataset

import torch
from torch.utils.data import Dataset, DataLoader

def encode(text, word2idx, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [word2idx.get(t, word2idx["<UNK>"]) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids = ids + [word2idx["<PAD>"]] * (max_len - len(ids))
    return ids

class MCQDataset(Dataset):
    """
    Each row -> 5 option sequences (prompt+A, prompt+B, ..., prompt+E)
    Label = index (0-4) of correct option among A-E
    """
    def __init__(self, df, word2idx, max_len=MAX_LEN, has_labels=True):
        self.rows = df.reset_index(drop=True)
        self.word2idx = word2idx
        self.max_len = max_len
        self.has_labels = has_labels
        self.option_cols = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        option_ids = []
        for col in self.option_cols:
            text = combine_text(row['prompt'], row[col])
            ids = encode(text, self.word2idx, self.max_len)
            option_ids.append(ids)

        option_tensor = torch.tensor(option_ids, dtype=torch.long)  # shape: (5, max_len)

        if self.has_labels:
            label = self.option_cols.index(row['answer'])  # 0-4
            return option_tensor, torch.tensor(label, dtype=torch.long)
        else:
            return option_tensor

# Quick sanity check on one sample
train_dataset = MCQDataset(train_split, word2idx)
sample_x, sample_y = train_dataset[0]
print("Sample option_tensor shape:", sample_x.shape)  # should be (5, 90)
print("Sample label:", sample_y.item())

Sample option_tensor shape: torch.Size([5, 90])
Sample label: 0


In [24]:
# BiLSTM model — Step 4: architecture (FIXED: added packing to handle padding correctly)
# Root cause of the earlier failure: post-padded sequences fed into an LSTM without
# packing means the LSTM reads through a long run of PAD tokens before producing its
# final hidden state, corrupting the representation (confirmed pattern, see e.g.
# github.com/keras-team/keras/issues/20898 — post-padding without masking breaks LSTM training)

import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class BiLSTMScorer(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, lengths):
        # x shape: (batch, 5, max_len), lengths shape: (batch, 5) — true token count per option
        batch_size, num_options, seq_len = x.shape
        x = x.view(batch_size * num_options, seq_len)
        lengths = lengths.view(batch_size * num_options)

        embedded = self.embedding(x)  # (batch*5, seq_len, embed_dim)

        # Clamp lengths to at least 1 to avoid pack_padded_sequence errors on empty sequences
        lengths_clamped = lengths.clamp(min=1)

        packed = pack_padded_sequence(embedded, lengths_clamped.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)  # hidden computed only over REAL tokens, not padding

        hidden_fwd = hidden[-2]
        hidden_bwd = hidden[-1]
        final_hidden = torch.cat([hidden_fwd, hidden_bwd], dim=1)

        final_hidden = self.dropout(final_hidden)
        scores = self.fc(final_hidden)
        scores = scores.view(batch_size, num_options)

        return scores

# Sanity check — now needs lengths too
model = BiLSTMScorer(vocab_size=VOCAB_SIZE)
dummy_lengths = (sample_x != 0).sum(dim=1).unsqueeze(0)  # count non-pad tokens per option
out = model(sample_x.unsqueeze(0), dummy_lengths)
print("Output shape:", out.shape)
print("Output (raw scores):", out)

Output shape: torch.Size([1, 5])
Output (raw scores): tensor([[-0.0688,  0.0294,  0.0834, -0.0127, -0.0005]],
       grad_fn=<ViewBackward0>)


In [25]:
# BiLSTM training — Step 5: training loop with packing, gradient clipping, lower LR
# Fixes applied: pack_padded_sequence (root cause fix), grad clipping (stability),
# lr 1e-3 -> 5e-4 (loss was oscillating, not smoothly decreasing, at the higher LR)

import torch.optim as optim
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

BATCH_SIZE = 32
EPOCHS = 15
LR = 5e-4
GRAD_CLIP = 1.0

train_dataset = MCQDataset(train_split, word2idx)
val_dataset = MCQDataset(val_split, word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = BiLSTMScorer(vocab_size=VOCAB_SIZE).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

run_bilstm = wandb.init(
    entity="23f2005025-dl-genai-project",
    project="dl-genai-project",
    name="day4-bilstm-scratch",
    job_type="from_scratch_model"
)

def get_lengths(x):
    # x: (batch, 5, max_len) -> (batch, 5) true token counts (non-pad)
    return (x != 0).sum(dim=2)

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            lengths = get_lengths(x)
            scores = model(x, lengths)
            loss = criterion(scores, y)
            total_loss += loss.item() * x.size(0)
            preds = scores.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0, 0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        lengths = get_lengths(x)

        optimizer.zero_grad()
        scores = model(x, lengths)
        loss = criterion(scores, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        preds = scores.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    train_loss /= total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

print("Training complete.")

Using device: cuda


wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json
wandb: uploading summary, console lines 116-134
wandb: 
wandb: Run history:
wandb: local_map3 ▁█
wandb: 
wandb: Run summary:
wandb:   approach minilm_embeddings
wandb: local_map3 0.39958
wandb: 
wandb: 🚀 View run day1-tfidf-baseline at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/1fwubbwp
wandb: ⭐️ View project at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260725_152533-1fwubbwp/logs
wandb: setting up run 0gx8r2xl
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260725_153919-0gx8r2xl
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run day4-bilstm-scratch
wandb: ⭐️ View project at https://wandb.ai/23

Epoch 1/15 | train_loss=1.5685 train_acc=0.3544 | val_loss=1.4985 val_acc=0.5100
Epoch 2/15 | train_loss=1.2837 train_acc=0.5350 | val_loss=0.9101 val_acc=0.6400
Epoch 3/15 | train_loss=0.7120 train_acc=0.7137 | val_loss=0.5309 val_acc=0.8550
Epoch 4/15 | train_loss=0.4156 train_acc=0.8625 | val_loss=0.3420 val_acc=0.8825
Epoch 5/15 | train_loss=0.2852 train_acc=0.9206 | val_loss=0.2667 val_acc=0.9300
Epoch 6/15 | train_loss=0.1760 train_acc=0.9569 | val_loss=0.1854 val_acc=0.9625
Epoch 7/15 | train_loss=0.1276 train_acc=0.9719 | val_loss=0.2312 val_acc=0.9450
Epoch 8/15 | train_loss=0.0934 train_acc=0.9819 | val_loss=0.0843 val_acc=0.9875
Epoch 9/15 | train_loss=0.0570 train_acc=0.9906 | val_loss=0.0431 val_acc=1.0000
Epoch 10/15 | train_loss=0.0287 train_acc=0.9969 | val_loss=0.0383 val_acc=0.9975
Epoch 11/15 | train_loss=0.0283 train_acc=0.9944 | val_loss=0.0219 val_acc=1.0000
Epoch 12/15 | train_loss=0.0097 train_acc=1.0000 | val_loss=0.0119 val_acc=1.0000
Epoch 13/15 | train_loss=

In [26]:
# BiLSTM diagnostic — Step 6: check for memorization via near-duplicate leakage
# Same logic as the RAG leakage check (Section 4 of Day 3 work): if val accuracy is
# only high because val rows have a near-identical twin in train_split, that's
# memorization, not generalization, and won't hold up on the leaderboard.

from difflib import SequenceMatcher

def has_near_duplicate(prompt, train_prompts, threshold=0.85):
    for tp in train_prompts:
        if SequenceMatcher(None, prompt, tp).ratio() > threshold:
            return True
    return False

train_prompts = train_split['prompt'].tolist()

val_split = val_split.copy()
val_split['has_dup'] = val_split['prompt'].apply(lambda p: has_near_duplicate(p, train_prompts))

print(f"Val rows WITH near-duplicate in train: {val_split['has_dup'].sum()} / {len(val_split)}")
print(f"Val rows WITHOUT near-duplicate in train: {(~val_split['has_dup']).sum()} / {len(val_split)}")

# Now evaluate model accuracy separately on each subset
model.eval()

def predict_batch(df_subset):
    ds = MCQDataset(df_subset, word2idx)
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            lengths = get_lengths(x)
            scores = model(x, lengths)
            preds = scores.argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(y.tolist())
    return all_preds, all_labels

dup_subset = val_split[val_split['has_dup']].reset_index(drop=True)
clean_subset = val_split[~val_split['has_dup']].reset_index(drop=True)

if len(dup_subset) > 0:
    preds, labels = predict_batch(dup_subset)
    acc_dup = sum(p == l for p, l in zip(preds, labels)) / len(labels)
    print(f"\nAccuracy on val rows WITH near-duplicate in train: {acc_dup:.4f} (n={len(dup_subset)})")

if len(clean_subset) > 0:
    preds, labels = predict_batch(clean_subset)
    acc_clean = sum(p == l for p, l in zip(preds, labels)) / len(labels)
    print(f"Accuracy on val rows WITHOUT near-duplicate in train: {acc_clean:.4f} (n={len(clean_subset)})")

Val rows WITH near-duplicate in train: 330 / 400
Val rows WITHOUT near-duplicate in train: 70 / 400

Accuracy on val rows WITH near-duplicate in train: 1.0000 (n=330)
Accuracy on val rows WITHOUT near-duplicate in train: 1.0000 (n=70)


In [27]:
# BiLSTM diagnostic — Step 7: check for shallow shortcut signals
# If accuracy is ~1.0 even on non-duplicate rows, the model may be exploiting a
# surface-level pattern (e.g. option length, position bias) rather than reasoning.

# Check 1: is the correct answer systematically longer/shorter than distractors?
def option_len_stats(df):
    correct_lens, wrong_lens = [], []
    for _, row in df.iterrows():
        for col in ['A', 'B', 'C', 'D', 'E']:
            length = len(str(row[col]).split())
            if col == row['answer']:
                correct_lens.append(length)
            else:
                wrong_lens.append(length)
    return correct_lens, wrong_lens

correct_lens, wrong_lens = option_len_stats(train)
print(f"Correct option avg length: {np.mean(correct_lens):.1f} words")
print(f"Wrong option avg length:   {np.mean(wrong_lens):.1f} words")

# Check 2: is there a positional bias (e.g. answer='A' way more often than others)?
print("\nAnswer letter distribution in train:")
print(train['answer'].value_counts(normalize=True).sort_index())

# Check 3: does the model's prediction correlate with option length even when wrong?
# Compare predicted option's length vs actual correct option's length on clean_subset
clean_preds, clean_labels = predict_batch(clean_subset)
option_cols = ['A', 'B', 'C', 'D', 'E']

pred_lens, actual_lens = [], []
for i, row in clean_subset.iterrows():
    pred_col = option_cols[clean_preds[clean_subset.index.get_loc(i)]]
    actual_col = row['answer']
    pred_lens.append(len(str(row[pred_col]).split()))
    actual_lens.append(len(str(row[actual_col]).split()))

print(f"\nOn clean subset — avg predicted-option length: {np.mean(pred_lens):.1f}, avg actual-correct-option length: {np.mean(actual_lens):.1f}")

Correct option avg length: 28.7 words
Wrong option avg length:   25.7 words

Answer letter distribution in train:
answer
A    0.1845
B    0.2450
C    0.2295
D    0.1790
E    0.1620
Name: proportion, dtype: float64

On clean subset — avg predicted-option length: 26.7, avg actual-correct-option length: 26.7


In [28]:
# BiLSTM — Step 8: compute local mAP@3 on val_split (to match RAG/MiniLM reporting format)

def get_top3_predictions(df, has_labels=True):
    ds = MCQDataset(df, word2idx, has_labels=has_labels)
    loader = DataLoader(ds, batch_size=32, shuffle=False)
    option_cols = ['A', 'B', 'C', 'D', 'E']
    all_top3 = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            x = batch[0] if has_labels else batch
            x = x.to(DEVICE)
            lengths = get_lengths(x)
            scores = model(x, lengths)
            top3_idx = scores.argsort(dim=1, descending=True)[:, :3].cpu()
            for row in top3_idx:
                all_top3.append([option_cols[i] for i in row])
    return all_top3

val_top3_preds = get_top3_predictions(val_split)
val_actual = val_split['answer'].tolist()

bilstm_map3 = mapk(val_actual, val_top3_preds, k=3)
print(f"BiLSTM local mAP@3 on val_split: {bilstm_map3:.4f}")

# Also log this alongside the training run for the required "3 runs compared on common metrics"
wandb.log({"final_local_map3": bilstm_map3})
wandb.finish()

wandb: updating run metadata


BiLSTM local mAP@3 on val_split: 1.0000


wandb: uploading history steps 15-15, summary, console lines 16-34
wandb: 
wandb: Run history:
wandb:            epoch ▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
wandb: final_local_map3 ▁
wandb:        train_acc ▁▃▅▇▇██████████
wandb:       train_loss █▇▄▃▂▂▂▁▁▁▁▁▁▁▁
wandb:          val_acc ▁▃▆▆▇▇▇████████
wandb:         val_loss █▅▃▃▂▂▂▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 15
wandb: final_local_map3 1
wandb:        train_acc 1
wandb:       train_loss 0.00269
wandb:          val_acc 1
wandb:         val_loss 0.0048
wandb: 
wandb: 🚀 View run day4-bilstm-scratch at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/0gx8r2xl
wandb: ⭐️ View project at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260725_153919-0gx8r2xl/logs


In [29]:
# Ensemble diagnostic — Step 1: do RAG and BiLSTM fail on the same rows or different rows?

# 'model' currently holds the trained BiLSTM (overwritten during training).
# Save that reference first, then reload MiniLM under a separate name for RAG.
trained_bilstm = model  # capture BiLSTM before we overwrite 'model'

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')  # restore for rag_predict_top3_v2's internal use

rag_val_preds = rag_predict_top3_v2(val_split, retrieve_k=10, rerank_top_n=3, exclude_leakage=False)

# Swap back to BiLSTM for its own inference
model = trained_bilstm
bilstm_val_preds = get_top3_predictions(val_split, has_labels=True)

val_actual = val_split['answer'].tolist()

rag_correct = [pred[0] == actual for pred, actual in zip(rag_val_preds, val_actual)]
bilstm_correct = [pred[0] == actual for pred, actual in zip(bilstm_val_preds, val_actual)]

both_correct = sum(r and b for r, b in zip(rag_correct, bilstm_correct))
only_rag_correct = sum(r and not b for r, b in zip(rag_correct, bilstm_correct))
only_bilstm_correct = sum(b and not r for r, b in zip(rag_correct, bilstm_correct))
both_wrong = sum(not r and not b for r, b in zip(rag_correct, bilstm_correct))

print(f"Both correct: {both_correct}")
print(f"Only RAG correct: {only_rag_correct}")
print(f"Only BiLSTM correct: {only_bilstm_correct}")
print(f"Both wrong: {both_wrong}")
print(f"\nRAG top-1 acc: {sum(rag_correct)/len(rag_correct):.4f}")
print(f"BiLSTM top-1 acc: {sum(bilstm_correct)/len(bilstm_correct):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Both correct: 328
Only RAG correct: 0
Only BiLSTM correct: 72
Both wrong: 0

RAG top-1 acc: 0.8200
BiLSTM top-1 acc: 1.0000


In [30]:
# Fine-tuned model — Step 1: setup and imports for AutoModelForMultipleChoice
# Using the HF standard multiple-choice architecture, purpose-built for N-option
# answer selection (same task shape as SWAG) rather than a hand-rolled classifier.

from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
import torch

MODEL_NAME = "distilbert-base-uncased"  # fast on T4, good fit for a small 2000-row dataset
                                          # (full fine-tuning of large models tends to overfit small data)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer for {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

# Quick tokenization sanity check on one example
sample_row = train_split.iloc[0]
sample_encoding = tokenizer(sample_row['prompt'], sample_row['A'], truncation=True, max_length=128)
print(f"\nSample token count for prompt+option A: {len(sample_encoding['input_ids'])}")
print(f"Decoded: {tokenizer.decode(sample_encoding['input_ids'])[:200]}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded tokenizer for distilbert-base-uncased
Vocab size: 30522

Sample token count for prompt+option A: 38
Decoded: [CLS] pick the best possible answer : what is the proposed name for the field that is responsible for cosmic inflation and the metric expansion of space? among the listed options. [SEP] inflaton [SEP]


In [31]:
# Fine-tuned model — Step 2: preprocess data into AutoModelForMultipleChoice's expected shape
# HF's multiple-choice collator expects, per example: num_choices copies of the prompt,
# each paired with one option, then labels as the correct option's index (0-4)

MAX_LEN_FT = 128  # HF encodes prompt+option pairs; 128 covers our earlier length stats comfortably
OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

def preprocess_for_multiple_choice(df, has_labels=True):
    """
    Returns a dict of lists ready for a HF Dataset:
    input_ids, attention_mask: each row is a list of 5 lists (one per option)
    labels: correct option index (0-4), only if has_labels
    """
    first_sentences = []  # prompt repeated 5x per row
    second_sentences = []  # the 5 options
    labels = []

    for _, row in df.iterrows():
        first_sentences.extend([row['prompt']] * 5)
        second_sentences.extend([row[col] for col in OPTION_COLS])
        if has_labels:
            labels.append(OPTION_COLS.index(row['answer']))

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LEN_FT,
        padding='max_length'
    )

    # Reshape flat (n_rows*5, seq_len) back into (n_rows, 5, seq_len)
    num_rows = len(df)
    result = {
        'input_ids': [tokenized['input_ids'][i*5:(i+1)*5] for i in range(num_rows)],
        'attention_mask': [tokenized['attention_mask'][i*5:(i+1)*5] for i in range(num_rows)],
    }
    if has_labels:
        result['labels'] = labels

    return result

# Preprocess train_split and val_split
train_processed = preprocess_for_multiple_choice(train_split, has_labels=True)
val_processed = preprocess_for_multiple_choice(val_split, has_labels=True)

print(f"Train examples: {len(train_processed['input_ids'])}")
print(f"Val examples: {len(val_processed['input_ids'])}")
print(f"Shape check — input_ids for row 0: {len(train_processed['input_ids'][0])} options x {len(train_processed['input_ids'][0][0])} tokens")
print(f"Sample label: {train_processed['labels'][0]}")

Train examples: 1600
Val examples: 400
Shape check — input_ids for row 0: 5 options x 128 tokens
Sample label: 0


In [32]:
# Fine-tuned model — Step 3: HF Dataset wrapper, custom collator, and model loading

from datasets import Dataset as HFDataset
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from typing import Optional, Union
import numpy as np

train_hf_dataset = HFDataset.from_dict(train_processed)
val_hf_dataset = HFDataset.from_dict(val_processed)

@dataclass
class DataCollatorForMultipleChoice:
    """
    Standard HF pattern for multiple-choice collation: flattens the (batch, num_choices, seq_len)
    structure to (batch*num_choices, seq_len) for padding, then reshapes back.
    """
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        has_labels = "labels" in features[0].keys() or "label" in features[0].keys()
        if has_labels:
            label_name = "labels" if "labels" in features[0].keys() else "label"
            labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if has_labels:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

# Load the pretrained model with a multiple-choice head
ft_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME} with multiple-choice head")
print(f"Total params: {sum(p.numel() for p in ft_model.parameters()):,}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForMultipleChoice LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded distilbert-base-uncased with multiple-choice head
Total params: 66,954,241


In [33]:
# Fine-tuned model — Step 4: Trainer setup and fine-tuning
# Fresh wandb.init() for this new run (per the established pattern)

from transformers import TrainingArguments, Trainer
import wandb

run_ft = wandb.init(
    entity="23f2005025-dl-genai-project",
    project="dl-genai-project",
    name="day5-distilbert-finetune",
    job_type="fine_tuned_model"
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert_ft_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,          # standard fine-tuning LR — much lower than BiLSTM's,
                                  # since we're adjusting pretrained weights, not training from random init
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,          # fine-tuning needs far fewer epochs than from-scratch training
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="wandb",
    run_name="day5-distilbert-finetune",
    logging_steps=20,
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_hf_dataset,
    eval_dataset=val_hf_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

wandb: setting up run e7v5losr
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260725_154406-e7v5losr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run day5-distilbert-finetune
wandb: ⭐️ View project at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project
wandb: 🚀 View run at https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/e7v5losr
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,2.373655,2.130185,0.737500
2,1.067067,0.894783,0.932500
3,0.573813,0.452858,0.962500
4,0.519392,0.350045,0.960000
5,0.437381,0.314182,0.967500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=500, training_loss=1.2457519550323486, metrics={'train_runtime': 289.0813, 'train_samples_per_second': 27.674, 'train_steps_per_second': 1.73, 'total_flos': 1324650362880000.0, 'train_loss': 1.2457519550323486, 'epoch': 5.0})

In [34]:
# Fine-tuned model — Step 5: compute local mAP@3 on val_split

import torch.nn.functional as F

def get_ft_top3_predictions(df, has_labels=True):
    processed = preprocess_for_multiple_choice(df, has_labels=has_labels)
    hf_ds = HFDataset.from_dict(processed)

    trainer_pred_output = trainer.predict(hf_ds)
    logits = trainer_pred_output.predictions  # shape: (n_rows, 5)

    top3_idx = np.argsort(-logits, axis=1)[:, :3]  # descending order, top 3
    top3_preds = [[OPTION_COLS[i] for i in row] for row in top3_idx]
    return top3_preds

val_ft_top3 = get_ft_top3_predictions(val_split, has_labels=True)
val_actual = val_split['answer'].tolist()

ft_map3 = mapk(val_actual, val_ft_top3, k=3)
print(f"DistilBERT fine-tuned local mAP@3 on val_split: {ft_map3:.4f}")

wandb.log({"final_local_map3": ft_map3})

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


DistilBERT fine-tuned local mAP@3 on val_split: 0.9833


In [35]:
# Fine-tuned model — Step 6: check for memorization via near-duplicate leakage
# Same diagnostic as BiLSTM's — confirm DistilBERT's high val score isn't riding
# on train/val duplicate pairs before trusting it

# Reuse val_split['has_dup'] computed earlier if still in memory; recompute if needed
if 'has_dup' not in val_split.columns:
    from difflib import SequenceMatcher
    train_prompts = train_split['prompt'].tolist()
    def has_near_duplicate(prompt, train_prompts, threshold=0.85):
        for tp in train_prompts:
            if SequenceMatcher(None, prompt, tp).ratio() > threshold:
                return True
        return False
    val_split = val_split.copy()
    val_split['has_dup'] = val_split['prompt'].apply(lambda p: has_near_duplicate(p, train_prompts))

dup_subset = val_split[val_split['has_dup']].reset_index(drop=True)
clean_subset = val_split[~val_split['has_dup']].reset_index(drop=True)

def ft_accuracy(df_subset):
    preds = get_ft_top3_predictions(df_subset, has_labels=True)
    actual = df_subset['answer'].tolist()
    top1_correct = sum(p[0] == a for p, a in zip(preds, actual))
    return top1_correct / len(actual)

if len(dup_subset) > 0:
    acc_dup = ft_accuracy(dup_subset)
    print(f"DistilBERT accuracy on val rows WITH near-duplicate in train: {acc_dup:.4f} (n={len(dup_subset)})")

if len(clean_subset) > 0:
    acc_clean = ft_accuracy(clean_subset)
    print(f"DistilBERT accuracy on val rows WITHOUT near-duplicate in train: {acc_clean:.4f} (n={len(clean_subset)})")

DistilBERT accuracy on val rows WITH near-duplicate in train: 0.9697 (n=330)


DistilBERT accuracy on val rows WITHOUT near-duplicate in train: 0.9571 (n=70)


In [36]:
# FINAL SUBMISSION — DistilBERT fine-tuned model (Day 5)
# This is the only cell in the notebook that writes submission.csv
run_final = wandb.init(
    entity="23f2005025-dl-genai-project",
    project="dl-genai-project",
    name="day5-distilbert-final-submission",
    job_type="fine_tuned_model"
)

print("Running DistilBERT inference on full test set (500 rows)...")
start = time.time()

test_ft_top3 = get_ft_top3_predictions(test, has_labels=False)

elapsed = time.time() - start
print(f"Took {elapsed:.1f}s for {len(test)} rows")

submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': [' '.join(pred) for pred in test_ft_top3]
})

submission.to_csv('submission.csv', index=False)
print("Final submission written:")
print(submission.head(10))

wandb.log({
    "final_approach": "distilbert_finetuned",
    "local_map3": ft_map3,
    "model_name": MODEL_NAME,
    "epochs": 5,
    "learning_rate": 2e-5
})
wandb.finish()

wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json; uploading config.yaml; uploading history steps 29-34, summary, console lines 10-17
wandb: uploading data
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▇███
wandb:               eval/loss █▃▂▁▁
wandb:            eval/runtime ▁█▅▂▁
wandb: eval/samples_per_second █▁▄▇█
wandb:   eval/steps_per_second █▁▄▇█
wandb:        final_local_map3 ▁
wandb:           test/accuracy ▇█▁
wandb:               test/loss ▂▁█
wandb:            test/runtime █▆▁
wandb: test/samples_per_second ▆█▁
wandb:                      +6 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.9675
wandb:               eval/loss 0.31418
wandb:            eval/runtime 4.5468
wandb: eval/samples_per_second 87.975
wandb:   eval/steps_per_second 5.498
wandb:        final_local_map3 0.98333
wandb:           test/acc

Running DistilBERT inference on full test set (500 rows)...


wandb: updating run metadata


Took 6.3s for 500 rows
Final submission written:
   ID Prediction
0   1      A C D
1   2      B E A
2   3      B E C
3   4      E D C
4   5      C A B
5   6      D B E
6   7      E A C
7   8      B A C
8   9      C D A
9  10      B E D


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading output.log; uploading wandb-summary.json
wandb: uploading history steps 0-1, summary, console lines 1-13
wandb: 
wandb: Run history:
wandb:                  epochs ▁
wandb:           learning_rate ▁
wandb:              local_map3 ▁
wandb:            test/runtime ▁
wandb: test/samples_per_second ▁
wandb:   test/steps_per_second ▁
wandb: 
wandb: Run summary:
wandb:                  epochs 5
wandb:          final_approach distilbert_finetuned...
wandb:           learning_rate 2e-05
wandb:              local_map3 0.98333
wandb:              model_name distilbert-base-unca...
wandb:            test/runtime 5.6937
wandb: test/samples_per_second 87.816
wandb:   test/steps_per_second 5.62
wandb: 
wandb: 🚀 View run day5-distilbert-final-submission at: https://wandb.ai/23f2005025-dl-genai-project/dl-genai-project/runs/tt2qo0hk
wandb: ⭐️ View project at: https://wandb.ai/23f2005025-dl-genai-project/d